In [1]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028FBFC8FC20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028FBD8CC9B0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

In [7]:
model_with_structure = model.with_structured_output(Movie,
                                                    method = "json_schema")
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000017C64183EC0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017C643E3710>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'response_format': {'type': 'json_schema', 'json_schema': {'name': 'Movie', 'schema': {'title': 'Movie', 'description': '', 'properties': {'ti

In [8]:
model.invoke("What is the movie 'Inception' about?")

AIMessage(content='**Inception** (2010), directed by Christopher\u202fNolan, is a sci‑fi heist thriller that explores the world of shared dreaming.\n\n### Core Premise\nThe film follows **Dom Cobb** (Leonardo\u202fDiCaprio), a skilled “extractor” who enters people’s subconscious while they dream in order to steal secrets. Cobb is haunted by the death of his wife, Mal, and the loss of his chance to return home to his children. He is offered a chance at redemption: instead of stealing an idea, he must **perform “inception”**—plant a new idea deep within a target’s mind so that the target believes it originated from themselves.\n\n### Plot Overview\n1. **The Job Offer** – Business magnate **Saito** (Ken Watanabe) hires Cobb to implant an idea in the mind of **Robert Fischer** (Cillian\u202fMurphy), the heir to a rival energy empire, to break up his father’s conglomerate.\n2. **Assembling the Team** – Cobb recruits a specialist crew:\n   - **Arthur** (Joseph\u202fGordon‑Levitt) – point‑man

In [9]:
response = model_with_structure.invoke("What is the movie 'Inception' about?")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The release year of the movie")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The rating of the movie")



model_with_structure = model.with_structured_output(Movie, method="json_schema", include_raw=True)
response = model_with_structure.invoke("What is the movie 'Inception' about?")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'reasoning_content': 'The user asks: "What is the movie \'Inception\' about?" The system expects a JSON response matching the "Movie" schema: fields title, year, director, rating. They ask about plot, but we must output JSON with those fields. We need to fill with appropriate info about Inception: title "Inception", year 2010, director "Christopher Nolan", rating maybe 8.8 (IMDb). Provide description? Not needed. Just output JSON object with those fields. Ensure compact JSON, no extra whitespace, valid. So: {"title":"Inception","year":2010,"director":"Christopher Nolan","rating":8.8}. That matches required fields. Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 178, 'prompt_tokens': 229, 'total_tokens': 407, 'completion_time': 0.379473292, 'completion_tokens_details': {'reasoning_tokens': 144}, 'prompt_time': 0.009851631, 'prompt_tok

# Nested Structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    """An actor with details."""
    name: str = Field(...,description="The name of the actor")
    role : str = Field(...,description="The role of the actor in the movie")


class MovieDetails(BaseModel):
    """A movie with details."""
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The release year of the movie")
    cast : list[Actor] = Field(...,description="The cast of the movie")
    genre: str = Field(...,description="The genre of the movie")
    budget: float = Field(...,description="The budget of the movie")




model_with_structure = model.with_structured_output(MovieDetails, method="json_schema", include_raw=True)
response = model_with_structure.invoke("What is the movie 'Inception' about?")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"cast":[{"name":"Leonardo DiCaprio","role":"Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Marion Cotillard","role":"Mal"},{"name":"Michael Caine","role":"Professor Stephen Miles"}],"genre":"Science fiction","budget":160000000}', additional_kwargs={'reasoning_content': 'The user asks: "What is the movie \'Inception\' about?" The system expects a JSON response matching the MovieDetails schema. Must provide fields: title, year, cast (array of objects with name, role), genre, budget. Must be compact JSON. Provide details about Inception. We need to fill with correct info: title "Inception", year 2010, genre "Science fiction", budget 160000000 (160 million). Cast: list of main actors: Leonardo DiCaprio as Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page (now Elliot Page) as Ariadne, Tom Hardy as Eames

# TypedDict

TypedDict provides a simpler alternative using python's built-in-typing.Ideal when we don't need runtime validation.

In [10]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]


model_with_typed_dict = model.with_structured_output(
    MovieDict,
    method="json_schema"
)

response = model_with_typed_dict.invoke(
    "Give me details of the movie Inception."
)

print(response)

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


In [11]:
from pydantic import  Field

class Actor(TypedDict):
    """An actor with details."""
    name: str = Field(...,description="The name of the actor")
    role : str = Field(...,description="The role of the actor in the movie")


class MovieDetails(TypedDict):
    """A movie with details."""
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The release year of the movie")
    cast : list[Actor] = Field(...,description="The cast of the movie")
    genre: str = Field(...,description="The genre of the movie")
    budget: float = Field(...,description="The budget of the movie")




model_with_structure = model.with_structured_output(MovieDetails, method="json_schema", include_raw=True)
response = model_with_structure.invoke("What is the movie 'Inception' about?")
response

{'raw': AIMessage(content='{"title":"Inception","year":2010,"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Marion Cotillard","role":"Mal"},{"name":"Michael Caine","role":"Professor Stephen Miles"}],"genre":"Science fiction","budget":160000000}', additional_kwargs={'reasoning_content': 'The user asks: "What is the movie \'Inception\' about?" The system expects a JSON response matching the MovieDetails schema. The schema requires fields: title (string), year (integer), cast (array of objects with name and role), genre (string), budget (number). The user asks for description, not specifically the fields. But we must output JSON with those fields. We need to fill with appropriate data for Inception. Provide title: "Inception", year: 2010, genre: "Science fiction", budget: maybe 160000000 (160 million). Provide ca

In [12]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data Classes

In [15]:
from pydantic import  Field, BaseModel
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(model=model, 
                     response_format=ContactInfo
                     )

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact information from: John Doe, john@example.com, 555-1234"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='555-1234'


In [16]:
result

{'messages': [HumanMessage(content='Extract contact information from: John Doe, john@example.com, 555-1234', additional_kwargs={}, response_metadata={}, id='db934758-e775-4ef1-a5e4-f9ba3f0693e5'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"555-1234"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from the given text. According to instructions, we must output ONLY the final JSON object adhering to ContactInfo schema. Must include name, email, phone. Provide compact JSON (no whitespace). So produce something like {"name":"John Doe","email":"john@example.com","phone":"555-1234"} Ensure all required fields present. Use compact format.'}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 235, 'total_tokens': 345, 'completion_time': 0.228635765, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.009439139, 'prompt_tokens_details': None, 'queue_time': 0.387294157, 'total_time':

In [17]:
# TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(model=model, 
                     response_format=ContactInfo
                     )



result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact information from: John Doe, john@example.com, 555-1234"}]
})

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '555-1234'}


In [18]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(model=model, 
                     response_format=ContactInfo)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact information from: John Doe, john@example.com, 555-1234"}]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='555-1234')
